# Factor-specific case lists with role-based sentence adjustments

This notebook lists verified charges that have both an explicit `starting_point` and an explicit `sentence_after_role` that differ in total months. Charges are grouped into five lists:

- Sentence: Starting/After Role Difference (all charges meeting the sentence-adjustment criterion)
- Aggravating: Role of the defendant
- Aggravating: Other (aggregated)
- Mitigating: Other (aggregated)
- Mitigating: Extreme youth

The output workbook contains one `All lists` sheet with every matching row, plus a separate sheet for each list.

In [14]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv

repo_root = Path.cwd().resolve()
if not (repo_root / 'featureExtraction').exists():
    repo_root = repo_root.parent

for env_path in (
    repo_root / 'featureExtraction' / '.env',
    repo_root / 'featureVerification' / '.env.local',
    repo_root / '.env',
):
    if env_path.exists():
        load_dotenv(env_path)

from evaluate_verified_sentences import get_collection, get_total_months

INFERRED_ROLE_SOURCE = 'Inferred as starting point since role adjustment not provided'

TARGET_FACTORS = [
    ('Sentence', 'Starting/After Role Difference'),
    ('Aggravating', 'Role of the defendant'),
    ('Aggravating', 'Other (aggregated)'),
    ('Mitigating', 'Other (aggregated)'),
    ('Mitigating', 'Extreme youth'),
]

SHEET_NAME_MAP = {
    ('Sentence', 'Starting/After Role Difference'): 'Starting-After Role Diff',
    ('Aggravating', 'Role of the defendant'): 'Aggravating - Role',
    ('Aggravating', 'Other (aggregated)'): 'Aggravating - Other',
    ('Mitigating', 'Other (aggregated)'): 'Mitigating - Other',
    ('Mitigating', 'Extreme youth'): 'Mitigating - Extreme youth',
}

COLUMN_ORDER = [
    'neutral_citation',
    'exclude_case',
    'trial_index',
    'Charge_no',
    'Defendant_id',
    'factor_category',
    'factor',
    'other_factor',
    'starting_point_total_months',
    'sentence_after_role_total_months',
    'difference_months',
    'drugs',
    'remarks',
]

verified_collection, _ = get_collection()
trial_query_conditions = [
    {
        'starting_point': {'$exists': True, '$ne': None},
        'sentence_after_role.source': {'$ne': INFERRED_ROLE_SOURCE},
    },
    {
        'aggravating_factors': {
            '$elemMatch': {'factor': 'Role of the defendant'},
        },
    },
    {
        'aggravating_factors': {
            '$elemMatch': {'factor': 'Other'},
        },
    },
    {
        'mitigating_factors': {
            '$elemMatch': {'factor': 'Other'},
        },
    },
    {
        'mitigating_factors': {
            '$elemMatch': {'factor': 'Extreme youth'},
        },
    },
]
query = {
    'is_verified': True,
    'trials.trials': {
        '$elemMatch': {
            '$or': trial_query_conditions,
        }
    },
}
projection = {
    'filename': 1,
    'judgement.neutral_citation': 1,
    'exclude': 1,
    'remarks': 1,
    'trials': 1,
}
docs = list(verified_collection.find(query, projection))


def format_drugs(trial: dict[str, Any]) -> str:
    parts = []
    for drug in trial.get('drugs') or []:
        drug_type = drug.get('drug_type')
        quantity = drug.get('quantity')
        if drug_type:
            parts.append(f'{drug_type}:{quantity}')
    return '; '.join(parts)


def is_explicit_sentence_after_role(sentence_after_role: dict[str, Any] | None) -> bool:
    if not sentence_after_role:
        return False
    source = sentence_after_role.get('source') or ''
    return source != INFERRED_ROLE_SOURCE


def matches_factor(
    factor_category: str, factor_display: str, trial: dict[str, Any]
) -> list[dict[str, Any]]:
    """Return matched factor dicts for the requested category/display label."""
    if factor_category == 'Aggravating':
        factors = trial.get('aggravating_factors') or []
        target = 'Role of the defendant' if factor_display == 'Role of the defendant' else 'Other'
    else:
        factors = trial.get('mitigating_factors') or []
        target = 'Extreme youth' if factor_display == 'Extreme youth' else 'Other'
    return [f for f in factors if f.get('factor') == target]


def build_row(
    doc: dict[str, Any],
    trial: dict[str, Any],
    index: int,
    category: str,
    display: str,
    other_factor: str | None,
    starting_total: int | None,
    after_role_total: int | None,
) -> dict[str, Any]:
    charge_type = trial.get('charge_type') or {}
    return {
        'neutral_citation': (doc.get('judgement') or {}).get('neutral_citation'),
        'exclude_case': bool(doc.get('exclude')),
        'trial_index': index,
        'Charge_no': charge_type.get('charge_no'),
        'Defendant_id': charge_type.get('defendant_id'),
        'factor_category': category,
        'factor': display,
        'other_factor': other_factor,
        'starting_point_total_months': starting_total,
        'sentence_after_role_total_months': after_role_total,
        'difference_months': abs(after_role_total - starting_total) if starting_total is not None and after_role_total is not None else None,
        'drugs': format_drugs(trial),
        'remarks': doc.get('remarks'),
    }


rows: list[dict[str, Any]] = []
sentence_diff_rows = 0
skipped_no_role_value = 0
skipped_no_difference = 0
extreme_youth_rows = 0
for doc in docs:
    trials = (doc.get('trials') or {}).get('trials') or []
    for index, trial in enumerate(trials):
        starting_point = trial.get('starting_point')
        sentence_after_role = trial.get('sentence_after_role')
        starting_total = get_total_months(starting_point) if starting_point else None
        explicit_sentence_after_role = is_explicit_sentence_after_role(sentence_after_role)
        after_role_total = get_total_months(sentence_after_role) if explicit_sentence_after_role else None
        has_sentence_adjustment = (
            starting_total is not None
            and after_role_total is not None
            and starting_total != after_role_total
        )

        if not starting_point or not explicit_sentence_after_role:
            skipped_no_role_value += 1
        elif starting_total == after_role_total:
            skipped_no_difference += 1

        if has_sentence_adjustment:
            rows.append(
                build_row(
                    doc,
                    trial,
                    index,
                    'Sentence',
                    'Starting/After Role Difference',
                    None,
                    starting_total,
                    after_role_total,
                )
            )
            sentence_diff_rows += 1

        for category, display in TARGET_FACTORS:
            if category == 'Sentence':
                continue
            matched_factors = matches_factor(category, display, trial)
            if not matched_factors:
                continue
            other_texts = [
                f.get('other_factor')
                for f in matched_factors
                if f.get('other_factor')
            ]
            other_factor = ' | '.join(dict.fromkeys(other_texts)) if other_texts else None
            rows.append(
                build_row(
                    doc,
                    trial,
                    index,
                    category,
                    display,
                    other_factor,
                    starting_total,
                    after_role_total,
                )
            )
            if category == 'Mitigating' and display == 'Extreme youth':
                extreme_youth_rows += 1

all_lists_df = pd.DataFrame(rows)
role_combined_df = None
if not all_lists_df.empty:
    all_lists_df = (
        all_lists_df
        .reindex(columns=COLUMN_ORDER)
        .sort_values(['factor_category', 'factor', 'neutral_citation', 'trial_index'])
        .reset_index(drop=True)
    )

    role_combined_mask = (
        (
            (all_lists_df['factor_category'] == 'Aggravating')
            & (all_lists_df['factor'] == 'Role of the defendant')
        )
        | (
            (all_lists_df['factor_category'] == 'Sentence')
            & (all_lists_df['factor'] == 'Starting/After Role Difference')
        )
    )
    if role_combined_mask.any():
        role_combined_df = all_lists_df[role_combined_mask].copy()
        role_combined_df['Factor: Role of the defendant'] = (
            (role_combined_df['factor_category'] == 'Aggravating')
            & (role_combined_df['factor'] == 'Role of the defendant')
        )
        role_combined_df['Factor: Starting/After Role Difference'] = (
            (role_combined_df['factor_category'] == 'Sentence')
            & (role_combined_df['factor'] == 'Starting/After Role Difference')
        )
        role_combined_df = (
            role_combined_df.groupby(
                ['neutral_citation', 'exclude_case', 'trial_index', 'Charge_no', 'Defendant_id'],
                as_index=False,
                sort=False,
            ).agg(
                {
                    'Factor: Role of the defendant': 'max',
                    'Factor: Starting/After Role Difference': 'max',
                    'starting_point_total_months': 'first',
                    'sentence_after_role_total_months': 'first',
                    'difference_months': 'first',
                    'drugs': 'first',
                    'remarks': 'first',
                }
            )
        )
        role_combined_df['Factor: Role of the defendant'] = role_combined_df['Factor: Role of the defendant'].astype(bool)
        role_combined_df['Factor: Starting/After Role Difference'] = role_combined_df['Factor: Starting/After Role Difference'].astype(bool)
        role_combined_order = [
            'neutral_citation',
            'exclude_case',
            'trial_index',
            'Charge_no',
            'Defendant_id',
            'Factor: Role of the defendant',
            'Factor: Starting/After Role Difference',
            'starting_point_total_months',
            'sentence_after_role_total_months',
            'difference_months',
            'drugs',
            'remarks',
        ]
        role_combined_df = role_combined_df[role_combined_order]

output_dir = repo_root / 'notebooks'
output_dir.mkdir(exist_ok=True)
output_path = output_dir / 'factor_role_and_other_review.xlsx'

with pd.ExcelWriter(output_path) as writer:
    all_lists_df.to_excel(writer, sheet_name='All lists', index=False)
    for category, display in TARGET_FACTORS:
        sheet_df = all_lists_df[
            (all_lists_df['factor_category'] == category)
            & (all_lists_df['factor'] == display)
        ].copy()
        sheet_name = SHEET_NAME_MAP[(category, display)]
        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
    if role_combined_df is not None:
        role_combined_df.to_excel(writer, sheet_name='Role Combined', index=False)

print(f'Processed {len(docs)} verified judgements')
print(f'Sentence starting/after role difference rows: {sentence_diff_rows}')
print(f'Skipped {skipped_no_role_value} trials with no explicit starting_point/sentence_after_role')
print(f'Skipped {skipped_no_difference} trials where starting_point equals sentence_after_role')
print(f'Wrote {len(all_lists_df)} matching factor rows to {output_path}')
print(f'Extreme youth rows written: {extreme_youth_rows}')
all_lists_df.head()

Processed 1050 verified judgements
Sentence starting/after role difference rows: 273
Skipped 761 trials with no explicit starting_point/sentence_after_role
Skipped 418 trials where starting_point equals sentence_after_role
Wrote 1130 matching factor rows to /Users/cxiang/Projects/drug-trafficing-sentence-predictor/notebooks/factor_role_and_other_review.xlsx
Extreme youth rows written: 14


,neutral_citation,exclude_case,trial_index,Charge_no,Defendant_id,factor_category,factor,other_factor,starting_point_total_months,sentence_after_role_total_months,difference_months,drugs,remarks
0,[2021] HKCFI 1718,False,0,1,1,Aggravating,Other (aggregated),Form 8 holder,162.0,162.0,0.0,Cocaine:304; Cannabis:4.98,While the specific details of the location the...
1,[2021] HKCFI 2055,False,0,1,1,Aggravating,Other (aggregated),They each entered Hong Kong with the specific ...,396.0,NaN,NaN,Cocaine:38554,The judge adopted a global approach for count ...
2,[2021] HKCFI 2055,False,1,1,2,Aggravating,Other (aggregated),They each entered Hong Kong with the specific ...,396.0,NaN,NaN,Cocaine:38554,The judge adopted a global approach for count ...
3,[2021] HKCFI 2055,False,2,2,1,Aggravating,Other (aggregated),They each entered Hong Kong with the specific ...,396.0,NaN,NaN,Cocaine:37120,The judge adopted a global approach for count ...
4,[2021] HKCFI 2055,False,3,2,2,Aggravating,Other (aggregated),They each entered Hong Kong with the specific ...,396.0,NaN,NaN,Cocaine:37120,The judge adopted a global approach for count ...
